# Dataset Beobank — Notebook de référence
## Documentation des 5 tables sources (Jour 1 / Jour 2 / Jour 3)

Ce notebook est **à part** : il ne contient aucun exercice, il documente le jeu de
données `../data/` tel qu'il est réellement lu dans les notebooks formateur/participant.
Consultez-le en cas de doute sur une colonne, un code, ou une relation entre tables.

**Fichiers documentés :** `CTR.csv`, `TIE.csv`, `TIE_ADR.csv`, `TIE_X_CTR.csv`, `TXN_X_CTR.csv`

**Convention de lecture commune** (utilisée partout dans la formation) :
```python
PARAMS = dict(sep=";", na_values=".", encoding="utf-8")
df = pd.read_csv("../data/NOM.csv", **PARAMS)
```
- `sep=";"` — les fichiers sont au format export SAS, séparateur point-virgule
- `na_values="."` — un `.` isolé signifie "valeur manquante" (convention SAS) → converti en `NaN`

## Setup — charger les 5 tables

In [1]:
import pandas as pd

PARAMS = dict(sep=";", na_values=".", encoding="utf-8")
DATA   = "../data"

ctr = pd.read_csv(f"{DATA}/CTR.csv",       **PARAMS)   # contrats
tie = pd.read_csv(f"{DATA}/TIE.csv",       **PARAMS)   # clients (tiers)
adr = pd.read_csv(f"{DATA}/TIE_ADR.csv",   **PARAMS)   # adresses des clients
txc = pd.read_csv(f"{DATA}/TIE_X_CTR.csv", **PARAMS)   # lien client ↔ contrat
txn = pd.read_csv(f"{DATA}/TXN_X_CTR.csv", **PARAMS)   # transactions

for nom, df in [("CTR",ctr),("TIE",tie),("TIE_ADR",adr),("TIE_X_CTR",txc),("TXN_X_CTR",txn)]:
    print(f"  {nom:10s} : {df.shape[0]:5d} lignes, {df.shape[1]:2d} colonnes")

  CTR        :   200 lignes, 11 colonnes
  TIE        :   100 lignes, 11 colonnes
  TIE_ADR    :   100 lignes, 20 colonnes
  TIE_X_CTR  :   200 lignes,  6 colonnes
  TXN_X_CTR  :  1260 lignes, 10 colonnes


## 1. CTR — Contrats

Une ligne = un contrat/compte bancaire Beobank. Table centrale : toutes les autres
tables s'y rattachent via `IDT_AC` (identifiant de compte) ou `REF_CTR_INN`.

In [ ]:
print(ctr.dtypes)
ctr.head(3)

**Dictionnaire de données — CTR**

| Colonne | Type | Description | Domaine observé |
|---|---|---|---|
| `IDT_AC` | int | Identifiant du compte (clé) | 200 valeurs uniques, ex. `65500004701` |
| `REF_CTR_INN` | int | Référence interne du contrat | — |
| `DAT_OUV_CTR` | date | Date d'ouverture du contrat | 2009 → 2026 |
| `COD_ECV_CTR` | int | Code statut du contrat | **voir note ci-dessous** |
| `DAT_ECV_CTR` | date | Date du dernier changement de statut | — |
| `DAT_CLO_CTR` | date | Date de clôture (si clôturé) | ~52% manquant (contrats non clôturés) |
| `COD_DEV` | str | Devise du compte | `EUR`, `USD`, `AUD`, `NOK` |
| `SLD_CTR` | float | Solde du compte | ~68% manquant dans cet extrait |
| `DAT_MAJ_SLD` | date | Date de dernière mise à jour du solde | — |
| `SLD_DSP` | float | Solde disponible | — |
| `MNT_INI` | float | Montant initial du contrat | — |

⚠️ **Note sur `COD_ECV_CTR`** — les notebooks Jour 1/2/3 utilisent un référentiel
pédagogique à 6 valeurs (`1`=Ouvert, `2`=En attente, `3`=Suspendu, `4`=Clôturé,
`5`=En résiliation, `6`=Résilié) pour enseigner les dictionnaires/formats. Dans
**cet échantillon de 200 contrats**, seuls les codes **4 et 6** sont réellement présents
(vérifié ci-dessous) : les exercices qui filtrent sur le statut `"1"` (ouvert) contre les
vraies données `ctr` renverront donc un résultat vide. Le code reste correct — c'est
la richesse de l'échantillon qui est limitée. Gardez-le en tête en salle.

In [2]:
print("Codes COD_ECV_CTR réellement présents :", sorted(ctr["COD_ECV_CTR"].dropna().unique().tolist()))
print(ctr["COD_ECV_CTR"].value_counts())

Codes COD_ECV_CTR réellement présents : [4, 6]
COD_ECV_CTR
4    105
6     95
Name: count, dtype: int64


## 2. TIE — Clients (tiers)

Une ligne = un client (particulier ou personne morale). Clé : `IDT_PI`.

In [3]:
print(tie.dtypes)
tie.head(3)

IDT_PI         int64
NUM_TIE        int64
COD_TYP_TIE    int64
COD_STA_FED    int64
DAT_STA_FED      str
DAT_PRE_CTR      str
DAT_DER_CTR      str
DAT_NAI          str
COD_LNG_CTR      str
DAT_DCS          str
COD_SEX          str
dtype: object


,IDT_PI,NUM_TIE,COD_TYP_TIE,COD_STA_FED,DAT_STA_FED,DAT_PRE_CTR,DAT_DER_CTR,DAT_NAI,COD_LNG_CTR,DAT_DCS,COD_SEX
0,655010234,2500003178436,2,4,2025-12-22,NaN,NaN,NaN,FR,NaN,NaN
1,655010248,2500003178512,1,3,NaN,NaN,NaN,2004-03-28,FR,NaN,M
2,655010249,2500003178544,1,1,2025-11-28,2025-11-28,NaN,1980-01-25,FR,NaN,M


**Dictionnaire de données — TIE**

| Colonne | Type | Description | Domaine observé |
|---|---|---|---|
| `IDT_PI` | int | Identifiant du tiers/client (clé) | 100 valeurs uniques |
| `NUM_TIE` | int | Numéro tiers (identifiant secondaire) | — |
| `COD_TYP_TIE` | int | Type de tiers | `1`=Personne physique (PP), `2`=Personne morale (PM) |
| `COD_STA_FED` | int | Statut fédéral (KYC/réglementaire) | `1` à `5` |
| `DAT_STA_FED` | date | Date de ce statut | — |
| `DAT_PRE_CTR` | date | Date du 1er contrat | ~93% manquant |
| `DAT_DER_CTR` | date | Date du dernier contrat | ~79% manquant |
| `DAT_NAI` | date | Date de naissance | 1936 → 2009 |
| `COD_LNG_CTR` | str | Langue du client | `FR`, `NL` |
| `DAT_DCS` | date | Date de décès (si applicable) | ~97% manquant (la plupart des clients sont vivants) |
| `COD_SEX` | str | Sexe | `M`, `F` |

In [4]:
print("COD_TYP_TIE :", tie["COD_TYP_TIE"].value_counts().to_dict())
print("COD_LNG_CTR :", tie["COD_LNG_CTR"].value_counts().to_dict())

COD_TYP_TIE : {1: 99, 2: 1}
COD_LNG_CTR : {'FR': 67, 'NL': 33}


## 3. TIE_ADR — Adresses des clients

Une ligne = l'adresse et les coordonnées d'un client. Clé : `IDT_PI` (même que TIE).
Table large (20 colonnes) et peu utilisée dans les exercices — surtout des champs
texte (nom, adresse, contact).

In [5]:
print(adr.dtypes)
adr[["IDT_PI","NOM_TIE","PRN","ADR_LIG1","NOM_VIL","COD_PAY_ISO","ADR_EMA"]].head(3)

IDT_PI               int64
NUM_TIE              int64
NOM_TIE                str
PRN                    str
ADR_LIG1               str
ADR_LIG2           float64
ADR_LIG3               str
ADR_LIG4               str
ADR_LIG5               str
ADR_LIG6               str
NUM_TEL_DOM_INL    float64
NUM_TEL_MOB_INL    float64
ADR_EMA                str
NOM_VIL                str
COD_IDT_NTN        float64
DAT_MAJ_ADR            str
LIB_TIT                str
COD_PST            float64
COD_PAY_ISO            str
NOM_CMU_NAI            str
dtype: object


,IDT_PI,NOM_TIE,PRN,ADR_LIG1,NOM_VIL,COD_PAY_ISO,ADR_EMA
0,655010234,CEBBCACFZAAAFZTIER,NaN,CEBBCACFZAAAFZTIER,BRUXELLES,BE,PRO@FREE.BE
1,655010248,VIRJXBF,NTASEIJZR,M NTASEIJZR VIRJXBF,BRUXELLES,BE,TTSAXL@E-I.COM
2,655010249,JANSSENS,BART,M BART JANSSENS,BRUSSEL,BE,BART.JANSSENS@GMAIL.COM


**Colonnes principales — TIE_ADR** (liste complète : voir `adr.columns` ci-dessus)

| Colonne | Description |
|---|---|
| `IDT_PI`, `NUM_TIE` | Clés, identiques à TIE |
| `NOM_TIE`, `PRN` | Nom et prénom |
| `ADR_LIG1` … `ADR_LIG6` | Lignes d'adresse (plusieurs colonnes vides selon la longueur de l'adresse) |
| `NUM_TEL_DOM_INL`, `NUM_TEL_MOB_INL` | Téléphones fixe / mobile |
| `ADR_EMA` | E-mail |
| `NOM_VIL`, `COD_PST`, `COD_PAY_ISO` | Ville, code postal, pays (ISO) |
| `LIB_TIT` | Civilité (Monsieur/Madame...) |

Beaucoup de colonnes texte contiennent des valeurs **anonymisées/générées** (ex. noms
sous forme de suites de lettres) : ne pas chercher de sens métier dans leur contenu,
seule la **structure** de la table est réaliste.

## 4. TIE_X_CTR — Lien client ↔ contrat

Table de jointure (many-to-many) : un client peut avoir plusieurs contrats, un
contrat peut avoir plusieurs titulaires. Clés : `IDT_PI` (→ TIE) et `IDT_AC` (→ CTR).

In [6]:
print(txc.dtypes)
txc.head(3)

IDT_PI         int64
IDT_AC         int64
NUM_TIE        int64
REF_CTR_INN    int64
FLG_PRE_TTL      str
COD_ROL_TTL      str
dtype: object


,IDT_PI,IDT_AC,NUM_TIE,REF_CTR_INN,FLG_PRE_TTL,COD_ROL_TTL
0,655010249,65500477817,2500003178544,25784077205,O,C
1,655010249,65500477836,2500003178544,25184076782,O,NaN
2,655010249,65500477838,2500003178544,25604083128,O,NaN


**Dictionnaire de données — TIE_X_CTR**

| Colonne | Type | Description | Domaine observé |
|---|---|---|---|
| `IDT_PI` | int | Identifiant client (→ TIE.IDT_PI) | — |
| `IDT_AC` | int | Identifiant compte (→ CTR.IDT_AC) | — |
| `NUM_TIE` | int | Numéro tiers | — |
| `REF_CTR_INN` | int | Référence contrat | — |
| `FLG_PRE_TTL` | str | Titulaire principal ? | `O`=Oui, `N`=Non |
| `COD_ROL_TTL` | str | Rôle du titulaire | `C` (Co-titulaire) ; ~28% manquant = titulaire principal seul |

In [7]:
# Un même client peut avoir plusieurs contrats
contrats_par_client = txc.groupby("IDT_PI").size()
print(f"Max de contrats pour un même client : {contrats_par_client.max()}")
print(f"Clients avec plus d'un contrat : {(contrats_par_client > 1).sum()} / {len(contrats_par_client)}")

Max de contrats pour un même client : 46
Clients avec plus d'un contrat : 8 / 10


## 5. TXN_X_CTR — Mouvements / transactions

Une ligne = un mouvement (opération) sur un compte. Clé étrangère : `IDT_AC` (→ CTR).
⚠️ **Cette table ne contient ni montant ni date de mouvement exploitable directement**
(uniquement des libellés d'opération, voir note ci-dessous).

In [8]:
print(txn.dtypes)
txn.head(3)

IDT_AC               int64
REF_CTR_INN          int64
NUM_FOL_XTR          int64
DAT_CRE_MVT_CPB        str
NUM_ORD_MVT_CPB      int64
COD_LNG_RIU            str
LIB_OPE_INL_1          str
LIB_OPE_INL_2          str
LIB_OPE_INL_3          str
LIB_OPE_INL_4      float64
dtype: object


,IDT_AC,REF_CTR_INN,NUM_FOL_XTR,DAT_CRE_MVT_CPB,NUM_ORD_MVT_CPB,COD_LNG_RIU,LIB_OPE_INL_1,LIB_OPE_INL_2,LIB_OPE_INL_3,LIB_OPE_INL_4
0,65500000611,29862221166,2005,2026-05-08,1,FR,REJ NATIONALE THA COMPTE SOLDE,NaN,NaN,NaN
1,65500000760,29882034602,6004,2026-04-27,1,FR,Domiciliation pour,SEPA THAIS,BBA,NaN
2,65500001861,29882205525,2004,2026-04-27,1,FR,BEOBANK BELGIUM,NaN,NaN,NaN


**Dictionnaire de données — TXN_X_CTR**

| Colonne | Type | Description | Domaine observé |
|---|---|---|---|
| `IDT_AC` | int | Identifiant compte (→ CTR.IDT_AC) | 1260 lignes, comptes pas tous dans CTR (voir note) |
| `REF_CTR_INN` | int | Référence contrat | — |
| `NUM_FOL_XTR` | int | Numéro de folio (relevé) | — |
| `DAT_CRE_MVT_CPB` | date | Date de création du mouvement | 2025-11 → 2026-05 |
| `NUM_ORD_MVT_CPB` | int | Numéro d'ordre du mouvement | — |
| `COD_LNG_RIU` | str | Langue du relevé | `FR`, `NL` |
| `LIB_OPE_INL_1..4` | str | Libellé de l'opération (jusqu'à 4 lignes de texte libre) | `LIB_OPE_INL_4` est vide à 100% |

⚠️ **Pas de montant/date de mouvement natifs** — `TXN_X_CTR.csv` ne fournit **aucune**
colonne de montant, et pas de date "de mouvement" distincte de `DAT_CRE_MVT_CPB`. Les
notebooks Jour 2 (fin) et Jour 3 **simulent** deux colonnes pour pouvoir enseigner
`rolling()`, `resample()`, les graphiques, etc. :
```python
rng = np.random.default_rng(42)
txn["DAT_MVT"] = pd.to_datetime(txn["DAT_CRE_MVT_CPB"])
txn["MNT_MVT"] = rng.normal(250, 400, size=len(txn)).round(2)
```
Ces deux colonnes sont **fictives** (graine fixe, reproductibles) — voir la cellule
Setup des notebooks formateur Jour 2/Jour 3 pour le code exact.

In [ ]:
print("Colonnes réellement présentes dans TXN_X_CTR.csv :", list(txn.columns))
print("\nAucune colonne de montant ni de date-de-mouvement distincte de DAT_CRE_MVT_CPB.")

## 6. Relations entre les tables

```
TIE ──(IDT_PI)── TIE_ADR        un client (TIE) a une adresse (TIE_ADR)
TIE ──(IDT_PI)── TIE_X_CTR ──(IDT_AC)── CTR     un client a un ou plusieurs contrats
CTR ──(IDT_AC)── TXN_X_CTR                       un contrat a des transactions
```

Chaîne de jointure complète utilisée dans les notebooks Jour 2/Jour 3 :
`CTR` → (`IDT_AC`) → `TIE_X_CTR` → (`IDT_PI`) → `TIE` → (`IDT_PI`) → `TIE_ADR`

In [ ]:
# Vérifier la couverture réelle des clés de jointure (proportion de lignes qui trouvent une correspondance)
print(f"TIE_X_CTR.IDT_AC trouvé dans CTR.IDT_AC : {txc['IDT_AC'].isin(ctr['IDT_AC']).mean():.0%}")
print(f"TIE_X_CTR.IDT_PI trouvé dans TIE.IDT_PI : {txc['IDT_PI'].isin(tie['IDT_PI']).mean():.0%}")
print(f"TIE.IDT_PI trouvé dans TIE_ADR.IDT_PI   : {tie['IDT_PI'].isin(adr['IDT_PI']).mean():.0%}")
print(f"TXN_X_CTR.IDT_AC trouvé dans CTR.IDT_AC : {txn['IDT_AC'].isin(ctr['IDT_AC']).mean():.0%}")

⚠️ **Note sur `TXN_X_CTR` ↔ `CTR`** — seule une minorité des transactions référence un
compte présent dans l'extrait `CTR.csv` (200 contrats). `TXN_X_CTR.csv` couvre un
périmètre de comptes plus large que l'extrait `CTR.csv` fourni. Conséquence pratique :
une jointure `ctr` ↔ `txn` sur `IDT_AC` perd la majorité des lignes de `txn`. Les
notebooks Jour 2/3 qui travaillent sur `txn` seul (sans jointure vers `ctr`) ne sont pas
concernés par cette limitation.

## 7. Conventions et limitations — récapitulatif

| Point | Détail |
|---|---|
| Séparateur | `;` (export SAS) |
| Valeur manquante | `.` dans le fichier → `NaN` avec `na_values="."` |
| Encodage | `utf-8` |
| Dates | Format `AAAA-MM-JJ` la plupart du temps ; `TIE_ADR.DAT_MAJ_ADR` au format `JJMONAAAA` (ex. `24NOV2025`) → utiliser `format="mixed"` |
| `COD_ECV_CTR` | Référentiel pédagogique à 6 valeurs, mais seuls `4` et `6` présents dans cet extrait CTR |
| `TXN_X_CTR` | Pas de montant/date de mouvement natifs → `MNT_MVT`/`DAT_MVT` **simulés** dans les notebooks Jour 2/3 |
| `TXN_X_CTR` ↔ `CTR` | Faible taux de correspondance sur `IDT_AC` (périmètres différents) |
| Identifiants | `IDT_AC`/`IDT_PI` réels sont de grands entiers (ex. `65500004701`) — les `"AC00001"` utilisés en Module 1 sont des exemples pédagogiques fictifs, pas des vrais identifiants du fichier |
| Champs texte (`TIE_ADR`) | Noms/adresses anonymisés/générés — structure réaliste, contenu sans signification métier |